In [2]:
# ==================================================
# 1.1 EXTRACTING THE IMF FORECAST FOR IRELAND (2024)
# ==================================================

# Importing pandas for working with Excel datasets
import pandas as pd

# Reading the historical IMF GDP growth forecasts
# Loading only the columns needed for our 2024 example
imf_forecasts = pd.read_excel(
    "WEOhistorical.xlsx",
    sheet_name="ngdp_rpch",
    usecols=[
        "country",
        "ISOAlpha_3Code",
        "year",
        "F2023ngdp_rpch"
    ],
    na_values="."
)

# Selecting the IMF forecast for Ireland's GDP growth in 2024
# Filtering the dataset using Ireland's country code and target year
ireland_forecast = imf_forecasts[
    (imf_forecasts["ISOAlpha_3Code"] == "IRL") &
    (imf_forecasts["year"] == 2024)
]

# Displaying Ireland's historical IMF forecast for 2024
print("IMF forecast for Ireland's 2024 GDP growth:")
display(ireland_forecast)


IMF forecast for Ireland's 2024 GDP growth:


,country,ISOAlpha_3Code,year,F2023ngdp_rpch
1092,Ireland,IRL,2024,3.349143


In [3]:
# ==================================================
# 1.2 EXTRACTING IMF HISTORICAL FORECASTS (2021-2024)
# ==================================================

# Reading the historical IMF GDP growth forecasts
# Loading the country information and four forecast columns
imf_all_forecasts = pd.read_excel(
    "WEOhistorical.xlsx",
    sheet_name="ngdp_rpch",
    usecols=[
        "country",
        "ISOAlpha_3Code",
        "year",
        "F2020ngdp_rpch",
        "F2021ngdp_rpch",
        "F2022ngdp_rpch",
        "F2023ngdp_rpch"
    ],
    na_values="."
)

# Matching each prediction year with the previous autumn's forecast
forecast_mapping = {
    2021: "F2020ngdp_rpch",
    2022: "F2021ngdp_rpch",
    2023: "F2022ngdp_rpch",
    2024: "F2023ngdp_rpch"
}

# Creating an empty list for storing the extracted forecasts
forecast_list = []

# Extracting the relevant IMF forecast for each target year
for target_year, column in forecast_mapping.items():

    # Selecting the rows for the year being predicted
    year_data = imf_all_forecasts[
        imf_all_forecasts["year"] == target_year
    ].copy()

    # Keeping the country details and selected forecast
    year_data = year_data[[
        "country",
        "ISOAlpha_3Code",
        "year",
        column
    ]]

    # Renaming the columns for consistency
    year_data = year_data.rename(columns={
        "country": "Country Name",
        "ISOAlpha_3Code": "Country Code",
        "year": "target_year",
        column: "imf_forecast"
    })

    # Adding the extracted forecasts to our list
    forecast_list.append(year_data)

# Combining the four years into one dataset
imf_forecast_data = pd.concat(
    forecast_list,
    ignore_index=True
)

# Converting forecast values into numeric format
# Treating missing or invalid values as NaN
imf_forecast_data["imf_forecast"] = pd.to_numeric(
    imf_forecast_data["imf_forecast"],
    errors="coerce"
)

# Selecting Ireland's forecasts for checking our results
ireland_forecasts = imf_forecast_data[
    imf_forecast_data["Country Code"] == "IRL"
]

# Displaying Ireland's four historical IMF forecasts
print("Ireland's historical IMF forecasts:")
display(ireland_forecasts)

# Counting available IMF forecasts for each target year
print("Available IMF forecasts by year:")
display(
    imf_forecast_data.groupby("target_year")[
        "imf_forecast"
    ].count()
)

Ireland's historical IMF forecasts:


,Country Name,Country Code,target_year,imf_forecast
24,Ireland,IRL,2021,4.942971
225,Ireland,IRL,2022,3.537945
426,Ireland,IRL,2023,3.999857
627,Ireland,IRL,2024,3.349143


Available IMF forecasts by year:


,imf_forecast
target_year,
2021,197
2022,196
2023,196
2024,195


In [4]:
# ==================================================
# 1.3 MATCHING IMF FORECASTS WITH ACTUAL GDP GROWTH
# ==================================================

# Reading our previously prepared GDP testing dataset
testing_data = pd.read_csv("GDP_Testing_2021_2024.csv")

# Selecting country information and actual GDP growth values
# Keeping the country codes and target years for matching
actual_gdp = testing_data[[
    "Country Name",
    "Country Code",
    "target_year",
    "next_year_gdp_growth"
]].copy()

# Renaming the GDP growth column for easier understanding
actual_gdp = actual_gdp.rename(columns={
    "next_year_gdp_growth": "actual_gdp_growth"
})

# Correcting the country codes that differ between IMF and World Bank
country_code_mapping = {
    "KOS": "XKX",
    "WBG": "PSE"
}

# Creating a separate copy of the IMF forecast dataset
imf_for_matching = imf_forecast_data.copy()

# Applying the country code corrections
imf_for_matching["Country Code"] = (
    imf_for_matching["Country Code"].replace(
        country_code_mapping
    )
)

# Selecting only the IMF columns needed for merging
imf_for_matching = imf_for_matching[[
    "Country Code",
    "target_year",
    "imf_forecast"
]]

# Merging the actual GDP growth values with IMF forecasts
# Matching the observations by country code and target year
# Keeping all rows from our GDP testing dataset
imf_comparison = pd.merge(
    actual_gdp,
    imf_for_matching,
    on=["Country Code", "target_year"],
    how="left",
    validate="one_to_one"
)

# Displaying Ireland's actual GDP growth and IMF forecasts
print("Ireland: Actual GDP Growth vs IMF Forecast")
display(
    imf_comparison[
        imf_comparison["Country Code"] == "IRL"
    ]
)

# Checking the number of observations after merging
print("Total rows:", len(imf_comparison))

# Counting observations containing both actual GDP and IMF forecasts
matched_observations = imf_comparison[
    imf_comparison["actual_gdp_growth"].notna() &
    imf_comparison["imf_forecast"].notna()
]

print("Matched observations:", len(matched_observations))

Ireland: Actual GDP Growth vs IMF Forecast


,Country Name,Country Code,target_year,actual_gdp_growth,imf_forecast
360,Ireland,IRL,2021,16.258188,4.942971
361,Ireland,IRL,2022,7.511139,3.537945
362,Ireland,IRL,2023,-2.536374,3.999857
363,Ireland,IRL,2024,2.599467,3.349143


Total rows: 868
Matched observations: 752


In [5]:
# ==================================================
# 1.4 CHECKING THE IMF BENCHMARK DATASET
# ==================================================

# Checking the number of missing actual GDP growth values
missing_actual = imf_comparison["actual_gdp_growth"].isna().sum()

# Checking the number of missing IMF forecasts
missing_forecasts = imf_comparison["imf_forecast"].isna().sum()

# Displaying the missing-value counts
print("Missing actual GDP values:", missing_actual)
print("Missing IMF forecasts:", missing_forecasts)

# Checking the number of matched observations for each year
# Counting only rows containing both actual GDP and IMF forecasts
matched_by_year = matched_observations.groupby(
    "target_year"
).size()

# Displaying the number of matched observations by year
print("\nMatched observations by year:")
display(matched_by_year)

# Checking for duplicate country-year combinations
# Identifying rows with the same country code and target year
duplicate_count = imf_comparison.duplicated(
    subset=["Country Code", "target_year"]
).sum()

# Displaying the number of duplicate observations
print("\nDuplicate country-year observations:", duplicate_count)

Missing actual GDP values: 45
Missing IMF forecasts: 104

Matched observations by year:


,0
target_year,
2021,189
2022,188
2023,188
2024,187



Duplicate country-year observations: 0


In [6]:
# ==================================================
# 1.5 SAVING THE IMF BENCHMARK DATASET
# ==================================================

# Creating a copy of the complete IMF comparison dataset
imf_benchmark_full = imf_comparison.copy()

# Sorting the observations by country and target year
imf_benchmark_full = imf_benchmark_full.sort_values(
    by=["Country Code", "target_year"]
)

# Saving the complete dataset, including missing values
imf_benchmark_full.to_csv(
    "IMF_Benchmark_Full_2021_2024.csv",
    index=False
)

# Selecting observations containing both actual GDP growth
# and historical IMF forecasts
imf_benchmark_matched = imf_benchmark_full[
    imf_benchmark_full["actual_gdp_growth"].notna() &
    imf_benchmark_full["imf_forecast"].notna()
].copy()

# Saving only the matched observations
imf_benchmark_matched.to_csv(
    "IMF_Benchmark_Matched_2021_2024.csv",
    index=False
)

# Displaying the number of saved observations
print("Full benchmark rows:", len(imf_benchmark_full))
print("Matched benchmark rows:", len(imf_benchmark_matched))

# Downloading both CSV files to my computer
from google.colab import files

files.download("IMF_Benchmark_Full_2021_2024.csv")
files.download("IMF_Benchmark_Matched_2021_2024.csv")

Full benchmark rows: 868
Matched benchmark rows: 752


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>